In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [ ]:
# Starting with fetching the links from the website
links = fetch_website_links("https://arnav.works/")
links

['#main',
 '#main',
 '#work',
 '#experience',
 '#certs',
 '#community',
 '#skills',
 '#contact',
 '#contact',
 '/Arnav_Gupta_Resume.pdf',
 'https://github.com/ARNAV2684',
 'https://www.linkedin.com/in/arnav-gupta-2b8b202b2/',
 'https://flickstat.com',
 'https://flickstat.com',
 'https://flickstat.com',
 'https://github.com/ARNAV2684/DataPro',
 'https://github.com/ARNAV2684/DataPro',
 'https://www.credly.com/badges/c90d5da9-c36d-4fe6-ab7f-cbf27ecc7b6c',
 'https://www.credly.com/badges/3e09954b-26fe-4e92-a2af-6732227ee24b',
 'mailto:arug2004@gmail.com',
 'https://github.com/ARNAV2684',
 'https://www.linkedin.com/in/arnav-gupta-2b8b202b2/',
 '/Arnav_Gupta_Resume.pdf']

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://arnav.works/"))


Here is the list of links on the website https://arnav.works/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main
#main
#work
#experience
#certs
#community
#skills
#contact
#contact
/Arnav_Gupta_Resume.pdf
https://github.com/ARNAV2684
https://www.linkedin.com/in/arnav-gupta-2b8b202b2/
https://flickstat.com
https://flickstat.com
https://flickstat.com
https://github.com/ARNAV2684/DataPro
https://github.com/ARNAV2684/DataPro
https://www.credly.com/badges/c90d5da9-c36d-4fe6-ab7f-cbf27ecc7b6c
https://www.credly.com/badges/3e09954b-26fe-4e92-a2af-6732227ee24b
mailto:arug2004@gmail.com
https://github.com/ARNAV2684
https://www.linkedin.com/in/arnav-gupta-2b8b202b2/
/Arnav_Gupta_Resume.pdf


In [10]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [11]:
select_relevant_links("https://arnav.works/")

{'links': [{'type': 'section anchor', 'url': 'https://arnav.works/#work'},
  {'type': 'section anchor', 'url': 'https://arnav.works/#experience'},
  {'type': 'section anchor', 'url': 'https://arnav.works/#certs'},
  {'type': 'section anchor', 'url': 'https://arnav.works/#community'},
  {'type': 'section anchor', 'url': 'https://arnav.works/#skills'},
  {'type': 'section anchor', 'url': 'https://arnav.works/#contact'},
  {'type': 'resume', 'url': 'https://arnav.works/Arnav_Gupta_Resume.pdf'},
  {'type': 'github profile', 'url': 'https://github.com/ARNAV2684'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/in/arnav-gupta-2b8b202b2/'},
  {'type': 'github repository', 'url': 'https://github.com/ARNAV2684/DataPro'},
  {'type': 'credly badge',
   'url': 'https://www.credly.com/badges/c90d5da9-c36d-4fe6-ab7f-cbf27ecc7b6c'},
  {'type': 'credly badge',
   'url': 'https://www.credly.com/badges/3e09954b-26fe-4e92-a2af-6732227ee24b'}]}

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'product page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'open source', 'url': 'https://github.com/huggingface'},
  {'type': 'community', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

In [14]:
# Now making the brochure with the relevant links and content from the website
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
thinkingmachines/Inkling
Updated
2 days ago
•
16.4k
•
1.44k
baidu/Unlimited-OCR
Updated
1 day ago
•
2.24M
•
2.7k
prism-ml/Ternary-Bonsai-27B-gguf
Updated
5 days ago
•
432k
•
936
poolside/Laguna-S-2.1
Updated
about 4 hours ago
•
3.06k
•
378
prism-ml/Bonsai-27B-gguf
Updated
5 days ago
•


In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'\n    You are looking at a company called: HuggingFace\n    Here are the contents of its landing page and other relevant pages;\n    use this information to build a short brochure of the company in markdown without code blocks.\n\n\n    ## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nthinkingmachines/Inkling\nUpdated\n2 days ago\n•\n16.4k\n•\n1.44k\nbaidu/Unlimited-OCR\nUpdated\n1 d

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

## Who We Are  
Hugging Face is the AI community building the future of machine learning. We provide an open, collaborative platform where researchers, developers, and enterprises come together to create, share, and advance the state-of-the-art in AI.  

## What We Offer  
- **Models:** Access and contribute to over 2 million machine learning models spanning text, image, video, and more.  
- **Datasets:** Explore and share from a vast repository of over 500,000 datasets, curated for diverse AI applications.  
- **Spaces:** Run and showcase AI applications with the community through interactive spaces, hosting thousands of live projects.  
- **Buckets:** Secure and scalable storage for ML assets and datasets.  
- **HuggingChat:** Cutting-edge chat AI applications utilizing state-of-the-art conversational models.

## Our Platform  
Hugging Face is the home of machine learning collaboration. We empower the community to:  
- Host unlimited public models, datasets, and applications with ease.  
- Access tools and infrastructure to move faster in research and deployment.  
- Leverage our open-source stack to build on top of proven technologies.  
- Explore AI across all modalities — text, image, video — without limits.  

## Community & Culture  
At our core, Hugging Face is a vibrant, open-source community fueled by collaboration and shared passion. We cultivate an inclusive environment where AI researchers, engineers, and enthusiasts from around the world connect through:  
- Forums and Discord for discussions and help  
- GitHub repositories to contribute code and models  
- Regular blog posts, papers, and daily AI news sharing  
- Open events and challenges to accelerate innovation  

Our culture is built on openness, transparency, and the collective mission to democratize AI for everyone.

## Customers & Enterprise Solutions  
We serve a broad spectrum of customers, including startups, academic institutions, and large enterprises seeking to integrate AI into their products, services, and research. Our offerings include:  
- **Team & Enterprise plans** with dedicated support and collaboration features.  
- **Hugging Face PRO** for professional-grade infrastructure and optimized model deployment.  
- **Enterprise Support** providing personalized assistance to scale AI workloads.  
- **Inference Endpoints and Providers** for seamless integration of AI models into applications.  

## Career Opportunities  
Join us to be part of shaping the future of AI! We value curiosity, collaboration, and impact. Our teams work across research, engineering, design, and community management — all contributing to an open ecosystem that transforms how AI is developed and used globally.  

Explore careers with Hugging Face to work alongside top AI talent and contribute to projects that truly make a difference.

---

## Contact & Learn More  
- Visit: [huggingface.co](https://huggingface.co)  
- Join the community discussions on Discord and Forum  
- Follow our latest updates via Blog and GitHub  

---

**Hugging Face** – The AI community building the future of machine learning, together.  
Bright colors of innovation: Yellow (#FFD21E), Orange (#FF9D00), and Gray (#6B7280).  
Empowering the world to create, discover, and collaborate on AI better.

In [21]:
# We will now use streaming to see the text as it is generated, instead of waiting for the entire brochure to be generated before we see it.
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Company Brochure

---

## About Hugging Face  
Hugging Face is the AI community building the future of machine learning. It serves as a pioneering collaboration platform where researchers, developers, and enterprises come together to create, discover, and share models, datasets, and AI applications. The company empowers the machine learning community by hosting over 2 million models, 500k+ datasets, and 1 million+ applications—making it the home of open-source machine learning efforts worldwide.

---

## What We Offer  

### Collaboration Platform  
- Host and collaborate on unlimited public models, datasets, and applications.  
- Facilitate community-driven innovation at scale with easy access to a robust AI ecosystem.

### Extensive AI Resources  
- Browse and utilize over 2 million machine learning models across modalities including text, image, and video.  
- Access 500k+ curated datasets powering research and application development.

### Spaces and Apps  
- Run and share AI-powered applications seamlessly in Spaces—our cloud-based platform for deploying ML apps.  
- Explore featured AI projects ranging from voice chat tools to large language models running locally in browsers.

### Enterprise Solutions  
- Hugging Face PRO and specialized enterprise support designed to empower organizations to accelerate AI innovation and deployment.
- Inference Endpoints, Providers, and Storage Buckets for scalable and secure AI infrastructure.

---

## Our Community & Culture  
Hugging Face cultivates an open, inclusive, and collaborative culture that supports knowledge sharing and rapid experimentation. The company champions open source principles, fostering an environment where developers and researchers contribute freely to collective progress. Through active forums, Discord channels, blogs, daily research papers, and public datasheets, Hugging Face nurtures an engaged and vibrant AI community.

---

## Customers & Use Cases  
Hugging Face’s platform is trusted by AI researchers, developers, data scientists, startups, and global enterprises. Industries ranging from healthcare, robotics, NLP, vision, to autonomous vehicles benefit from the vast repository of models and datasets. Hugging Face empowers customers to move faster in ML development and adopt cutting-edge technologies seamlessly.

---

## Careers & Opportunities  
Join Hugging Face to be at the forefront of AI innovation. The company values talent passionate about open source, AI ethics, and scalable AI solutions. Career opportunities span engineering, research, community management, product development, and enterprise sales. Working at Hugging Face means contributing to technology that shapes the future and collaborating globally with thought leaders.

---

## Brand & Identity  
Hugging Face’s brand is synonymous with openness, innovation, and community-driven progress in AI. Its visual identity features bold yellow (#FFD21E) and orange (#FF9D00) tones that reflect energy and creativity, balanced by neutral grays (#6B7280). The logo and brand assets are openly available for community use, further embodying the spirit of sharing and collaboration.

---

## Get Involved  
- Explore AI models and datasets at [huggingface.co](https://huggingface.co)  
- Join the discussions on Discord and Forums  
- Contribute on GitHub  
- Follow the latest AI research and tools via daily blog posts and papers  

---

**Hugging Face** — The AI community building the future of machine learning, together.

---